# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29820\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra column for the age of married people

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [20]:
df['obese'] = df['bmi'] >= 30

In [21]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

### Converting boolean columns to numerical

In [22]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Reformatting column names

In [23]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [24]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [25]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [26]:
df['log_bmi'] = np.log(df['bmi'])

In [27]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [28]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,obese,high_avg_glucose_level,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,1,1,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,1,0,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,1,1,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,0,1,3.178054,5.159745


In [29]:
df.shape

(5110, 27)

## Model Building

### Splitting to training, validation, and testing sets

In [30]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [31]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29820\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### SMOTE Oversampling on Training Data

In [33]:
resampler = SMOTE()

In [34]:
X_train_res, y_train_res = resampler.fit_resample(X_train, y_train)

### Create base random forest classifier model

In [35]:
base_classifier = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

### Create grid for hyperparameter tuning

In [36]:
param_grid_xgb = {
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1, 3, 5, 10, 15, 20]
}

In [37]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [38]:
grid_xgb = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_xgb,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

In [39]:
# importance_df = pd.DataFrame({'Feature': X_train_res.columns, 'Importance': base_classifier.feature_importances_})
# importance_df = importance_df.sort_values(by='Importance', ascending=False)
# print(importance_df)

### Fitting base classifier on unsampled data

In [40]:
base_classifier.fit(X_train_res, y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [41]:
importances = base_classifier.feature_importances_

In [42]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Importance': importances}).sort_values(
    'Importance', ascending=False)

In [43]:
print(feature_imp_df)

                           Feature  Importance
11              work_type_govt_job    0.108286
19           smoking_status_smokes    0.096095
18     smoking_status_never_smoked    0.083709
14         work_type_self_employed    0.082813
16          smoking_status_unknown    0.082759
0                              age    0.076800
21                           obese    0.069979
13               work_type_private    0.063380
17  smoking_status_formerly_smoked    0.062333
8                            smoke    0.057771
15              work_type_children    0.055281
12          work_type_never_worked    0.043750
7                            urban    0.018382
6                             male    0.013618
3                     ever_married    0.013167
2                    heart_disease    0.012854
20                age_ever_married    0.011533
4                avg_glucose_level    0.009253
5                              bmi    0.007726
1                     hypertension    0.006328
22          h

In [44]:
retained_columns = ['work_type_self_employed', 'smoking_status_unknown', 'smoking_status_formerly_smoked', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age', 'work_type_govt_job', 'work_type_children', 'work_type_private']

In [45]:
X_train_res_retained = X_train_res[retained_columns]

In [46]:
X_val_retained = X_val[retained_columns]

### Create optimizing function for hyperparameter tuning with Optuna

In [47]:
MIN_PRECISION = 0.15

In [48]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 15),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params)
    
    model.fit(X_train_res_retained, y_train_res)
    
    y_pred_val = model.predict(X_val_retained)
    
    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    
    if precision < MIN_PRECISION:
        return 0
    return recall + (precision * 0.01)

In [49]:
# grid_xgb.fit(X_train_res_retained, y_train_res)

In [50]:
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

[I 2026-01-19 22:18:45,941] A new study created in memory with name: no-name-71740303-ff11-40c8-a090-2a4a89252b60


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-01-19 22:18:46,776] Trial 0 finished with value: 0.0 and parameters: {'max_depth': 9, 'learning_rate': 0.2536999076681772, 'n_estimators': 233, 'min_child_weight': 5, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.05808361216819946, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 0.6011150117432088, 'scale_pos_weight': 10.913016089144637}. Best is trial 0 with value: 0.0.
[I 2026-01-19 22:18:47,035] Trial 1 finished with value: 0.6233640458640458 and parameters: {'max_depth': 3, 'learning_rate': 0.2708160864249968, 'n_estimators': 258, 'min_child_weight': 2, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'gamma': 0.3042422429595377, 'reg_alpha': 0.5247564316322378, 'reg_lambda': 0.43194501864211576, 'scale_pos_weight': 5.077207962772587}. Best is trial 1 with value: 0.6233640458640458.
[I 2026-01-19 22:18:47,325] Trial 2 finished with value: 0.5156974215594905 and parameters: {'max_depth': 14, 'learning_rate': 0.0160

In [51]:
# best_params_xgb = grid_xgb.best_params_

In [52]:
best_params_xgb = study_xgb.best_params

In [53]:
print(best_params_xgb)

{'max_depth': 15, 'learning_rate': 0.10511123501271838, 'n_estimators': 118, 'min_child_weight': 6, 'subsample': 0.6017408246664445, 'colsample_bytree': 0.5234492032064415, 'gamma': 0.7885198801145432, 'reg_alpha': 0.9417877267198208, 'reg_lambda': 0.681993125998142, 'scale_pos_weight': 5.575452929490148}


In [54]:
final_classifier = xgb.XGBClassifier(**best_params_xgb, random_state=42, eval_metric='logloss')

In [55]:
final_classifier.fit(X_train_res_retained, y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5234492032064415, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=0.7885198801145432, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.10511123501271838,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=15, max_leaves=None,
              min_child_weight=6, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=118, n_jobs=None,
              num_parallel_tree=None, ...)

In [56]:
y_pred_val = final_classifier.predict(X_val_retained)

In [57]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [58]:
print(confusion_matrix(y_val, y_pred_val))

[[593 136]
 [  9  28]]


In [59]:
print(accuracy_score(y_val, y_pred_val))

0.8107049608355091


In [60]:
print(precision_score(y_val, y_pred_val))

0.17073170731707318


In [61]:
print(recall_score(y_val, y_pred_val))

0.7567567567567568


In [62]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.44871794871794873


### Threshold tuning

In [63]:
best_threshold = 0.5

In [64]:
best_precision_score = 0

In [65]:
best_recall_score = 0

In [66]:
best_f2_score = 0

In [67]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.32015065913371
Threshold 0.15000000000000002 has F2 score 0.3225806451612903
Threshold 0.20000000000000004 has F2 score 0.34521158129175944
Threshold 0.25000000000000006 has F2 score 0.35799522673031026
Threshold 0.30000000000000004 has F2 score 0.35443037974683544
Threshold 0.3500000000000001 has F2 score 0.3804347826086957
Threshold 0.40000000000000013 has F2 score 0.40229885057471265
Threshold 0.45000000000000007 has F2 score 0.4307692307692308
Threshold 0.5000000000000001 has F2 score 0.44871794871794873
Threshold 0.5500000000000002 has F2 score 0.4452054794520548
Threshold 0.6000000000000002 has F2 score 0.4121863799283154
Threshold 0.6500000000000001 has F2 score 0.41338582677165353
Threshold 0.7000000000000002 has F2 score 0.4329004329004329
Threshold 0.7500000000000002 has F2 score 0.3881278538812785
Threshold 0.8000000000000002 has F2 score 0.3015075376884422
Threshold 0.8500000000000002 has F2 score 0.17142857142857143
Threshold 0.9000000000000002

In [68]:
print(best_threshold)

0.5000000000000001


In [69]:
print(best_recall_score)

0.7567567567567568


### Retrain on training + validation set

In [70]:
X_train_full = pd.concat([X_train_res_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train_res, y_val], axis=0, ignore_index=True)

In [71]:
final_classifier.fit(X_train_full, y_train_full)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5234492032064415, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=0.7885198801145432, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.10511123501271838,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=15, max_leaves=None,
              min_child_weight=6, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=118, n_jobs=None,
              num_parallel_tree=None, ...)

### Evaluate on test set

In [72]:
X_test_retained = X_test[retained_columns]

In [73]:
y_pred_test = final_classifier.predict(X_test_retained)

In [74]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [75]:
print(confusion_matrix(y_test, y_pred_test))

[[590 139]
 [ 15  23]]


In [76]:
print(accuracy_score(y_test, y_pred_test))

0.7992177314211213


In [77]:
print(precision_score(y_test, y_pred_test))

0.1419753086419753


In [78]:
print(recall_score(y_test, y_pred_test))

0.6052631578947368


In [79]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.3662420382165605
